# Identificación de municipios vecinos — Selección manual

Este notebook permite seleccionar **manualmente** los municipios que se quieren considerar
como vecinos del municipio de interés, en lugar de detectarlos por adyacencia automática.

**Fuente:** Marco Geoestadístico Nacional (MGN) del INEGI.

**Flujo de trabajo:**
1. Ejecuta las celdas hasta la sección 3 (carga de datos).
2. Selecciona el municipio de interés en los menús desplegables.
3. Ejecuta la celda de la sección 4 para confirmar.
4. Consulta el **mapa de referencia** (sección 5) para identificar los municipios del área.
5. En la sección 6, agrega uno a uno los municipios vecinos que deseas incluir.
6. Ejecuta las celdas de visualización y exportación.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import contextily as ctx
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from shapely.geometry import box as shp_box

## 1. Configuración

In [ ]:
# Ruta al shapefile de municipios del MGN
RUTA_MGN = Path("C:/SCINCE 2020/00_NAL/cartografia/municipal.shp")

# Carpeta donde se guardan los resultados
RUTA_SALIDA = Path("../datos/vecinos")

# Factor de extensión del área de contexto (2 = 2 veces el tamaño del municipio en c/dirección)
FACTOR_CONTEXTO = 2.0
MIN_EXTENT_M    = 10_000  # extensión mínima del municipio en metros

## 2. Carga del Marco Geoestadístico Nacional

In [ ]:
assert RUTA_MGN.exists(), (
    f"No se encontró el shapefile en {RUTA_MGN}. "
    "Revisa la ruta en la celda de configuración."
)

municipios = gpd.read_file(RUTA_MGN)

municipios["CVE_ENT"] = municipios["CVEGEO"].astype(str).str.zfill(5).str[:2]
municipios["CVE_MUN"] = municipios["CVEGEO"].astype(str).str.zfill(5).str[2:]

print(f"Total de municipios cargados: {len(municipios)}")
print(f"CRS: {municipios.crs}")
municipios[["CVEGEO", "CVE_ENT", "CVE_MUN", "NOM_ENT", "NOMGEO"]].head(3)

## 3. Selección del municipio de interés

Elige primero el **estado** y luego el **municipio de interés**.

In [ ]:
estados = sorted(municipios["NOM_ENT"].dropna().unique().tolist())

w_estado = widgets.Dropdown(
    options=estados,
    description="Estado:",
    layout=widgets.Layout(width="420px"),
    style={"description_width": "80px"},
)

w_municipio = widgets.Dropdown(
    options=[],
    description="Municipio:",
    layout=widgets.Layout(width="420px"),
    style={"description_width": "80px"},
)

out_info = widgets.Output()

def actualizar_municipios(change):
    muns = sorted(
        municipios.loc[municipios["NOM_ENT"] == change["new"], "NOMGEO"]
        .dropna().unique().tolist()
    )
    w_municipio.options = muns
    w_municipio.value = muns[0] if muns else None

def mostrar_info(change):
    with out_info:
        clear_output()
        if w_municipio.value:
            fila = municipios[
                (municipios["NOM_ENT"] == w_estado.value) &
                (municipios["NOMGEO"]  == w_municipio.value)
            ].iloc[0]
            print(f"\u2714  {fila['NOMGEO']}, {fila['NOM_ENT']}  "
                  f"(CVEGEO: {fila['CVEGEO']})")

w_estado.observe(actualizar_municipios, names="value")
w_municipio.observe(mostrar_info, names="value")
actualizar_municipios({"new": estados[0]})

display(widgets.VBox([
    widgets.Label("Selecciona el municipio de interés:"),
    w_estado,
    w_municipio,
    out_info,
]))

## 4. Confirmar municipio de interés

> Ejecuta esta celda **después** de haber seleccionado el municipio en los dropdowns.

In [ ]:
mascara = (
    (municipios["NOM_ENT"] == w_estado.value) &
    (municipios["NOMGEO"]  == w_municipio.value)
)

assert mascara.sum() == 1, (
    f"Se encontraron {mascara.sum()} registros para "
    f"'{w_municipio.value}, {w_estado.value}'. "
    "El shapefile puede tener duplicados."
)

mun_interes = municipios[mascara].iloc[0]
idx_interes  = municipios.index[mascara][0]

print(f"Municipio de interés : {mun_interes['NOMGEO']}, {mun_interes['NOM_ENT']}")
print(f"CVEGEO               : {mun_interes['CVEGEO']}")

## 5. Mapa de referencia

El mapa muestra el municipio de interés y los municipios del área circundante con sus nombres y claves CVEGEO, para que puedas identificar cuáles quieres agregar como vecinos en la siguiente sección.

In [ ]:
geom = municipios.at[idx_interes, "geometry"]
minx, miny, maxx, maxy = geom.bounds
dx = max(maxx - minx, MIN_EXTENT_M)
dy = max(maxy - miny, MIN_EXTENT_M)

area_ext = shp_box(
    minx - FACTOR_CONTEXTO * dx, miny - FACTOR_CONTEXTO * dy,
    maxx + FACTOR_CONTEXTO * dx, maxy + FACTOR_CONTEXTO * dy,
)
ctx_mask   = municipios.geometry.intersects(area_ext) & (municipios.index != idx_interes)
referencia = municipios[ctx_mask].copy()

CRS_VIZ = "EPSG:3857"
mun_ref_plot = municipios.loc[[idx_interes]].to_crs(CRS_VIZ)
ref_plot     = referencia.to_crs(CRS_VIZ)

fig, ax = plt.subplots(figsize=(12, 10))

ref_plot.plot(ax=ax, color="#e8e8e8", edgecolor="#999", linewidth=0.5, alpha=0.9)
mun_ref_plot.plot(ax=ax, color="#c0392b", edgecolor="black", linewidth=1.2, zorder=5)

for _, row in ref_plot.iterrows():
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    ax.annotate(
        f"{row['NOMGEO']}\n({row['CVEGEO']})",
        xy=(cx, cy),
        fontsize=6.5, ha="center", va="center", color="#333",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.75, ec="#ccc")
    )

try:
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom="auto")
except Exception:
    pass

ax.legend(handles=[
    mpatches.Patch(color="#c0392b",
                   label=f"Municipio de interés: {mun_interes['NOMGEO']}"),
    mpatches.Patch(facecolor="#e8e8e8", edgecolor="#999",
                   label="Municipios del área (para referencia)"),
], loc="lower left", fontsize=9)
ax.set_title(
    f"Mapa de referencia: entorno de {mun_interes['NOMGEO']}, "
    f"{mun_interes['NOM_ENT']}",
    fontsize=12, fontweight="bold"
)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"\nMunicipios en el área de referencia: {len(referencia)}")
referencia[["CVEGEO", "CVE_ENT", "CVE_MUN", "NOM_ENT", "NOMGEO"]].reset_index(drop=True)

## 6. Selección manual de municipios vecinos

Usa el mapa de referencia de la sección anterior para identificar los municipios.
Agrégalos uno a uno usando los menús de abajo.

> Ejecuta la siguiente celda y usa los botones interactivos.

In [ ]:
lista_seleccionados = []  # [{"CVEGEO": ..., "NOM_ENT": ..., "NOMGEO": ...}, ...]

w_estado_vec = widgets.Dropdown(
    options=estados,
    description="Estado:",
    layout=widgets.Layout(width="420px"),
    style={"description_width": "80px"},
)

w_municipio_vec = widgets.Dropdown(
    options=[],
    description="Municipio:",
    layout=widgets.Layout(width="420px"),
    style={"description_width": "80px"},
)

out_lista = widgets.Output()

def actualizar_municipios_vec(change):
    muns = sorted(
        municipios.loc[municipios["NOM_ENT"] == change["new"], "NOMGEO"]
        .dropna().unique().tolist()
    )
    w_municipio_vec.options = muns
    w_municipio_vec.value = muns[0] if muns else None

w_estado_vec.observe(actualizar_municipios_vec, names="value")
actualizar_municipios_vec({"new": estados[0]})

def _mostrar_lista():
    if not lista_seleccionados:
        print("(Lista vacía — agrega municipios con el botón de arriba)")
        return
    print(f"Vecinos seleccionados ({len(lista_seleccionados)}):")
    for i, s in enumerate(lista_seleccionados, 1):
        print(f"  {i}. {s['NOMGEO']}, {s['NOM_ENT']}  (CVEGEO: {s['CVEGEO']})")

def agregar_vecino(btn):
    nom_ent = w_estado_vec.value
    nomgeo  = w_municipio_vec.value
    if not nom_ent or not nomgeo:
        return
    filas = municipios[
        (municipios["NOM_ENT"] == nom_ent) & (municipios["NOMGEO"] == nomgeo)
    ]
    if filas.empty:
        return
    cvegeo = filas["CVEGEO"].values[0]
    with out_lista:
        clear_output()
        if cvegeo == mun_interes["CVEGEO"]:
            _mostrar_lista()
            print("\u26a0  Ese es el municipio de interés; no puede ser vecino de sí mismo.")
            return
        if any(s["CVEGEO"] == cvegeo for s in lista_seleccionados):
            _mostrar_lista()
            print(f"\u26a0  '{nomgeo}' ya está en la lista.")
            return
        lista_seleccionados.append(
            {"CVEGEO": cvegeo, "NOM_ENT": nom_ent, "NOMGEO": nomgeo}
        )
        _mostrar_lista()

def quitar_ultimo(btn):
    with out_lista:
        clear_output()
        if lista_seleccionados:
            eliminado = lista_seleccionados.pop()
            print(f"Eliminado: {eliminado['NOMGEO']}, {eliminado['NOM_ENT']}")
        _mostrar_lista()

def limpiar_lista(btn):
    lista_seleccionados.clear()
    with out_lista:
        clear_output()
        _mostrar_lista()

btn_agregar = widgets.Button(
    description="+ Agregar",
    button_style="primary",
    layout=widgets.Layout(width="140px"),
)
btn_quitar = widgets.Button(
    description="Quitar último",
    button_style="warning",
    layout=widgets.Layout(width="140px"),
)
btn_limpiar = widgets.Button(
    description="Limpiar lista",
    button_style="danger",
    layout=widgets.Layout(width="140px"),
)

btn_agregar.on_click(agregar_vecino)
btn_quitar.on_click(quitar_ultimo)
btn_limpiar.on_click(limpiar_lista)

with out_lista:
    _mostrar_lista()

display(widgets.VBox([
    widgets.Label("Selecciona los municipios vecinos uno a uno:"),
    w_estado_vec,
    w_municipio_vec,
    widgets.HBox([btn_agregar, btn_quitar, btn_limpiar]),
    out_lista,
]))

## 7. Confirmar lista de vecinos

> Ejecuta esta celda **después** de terminar de agregar los municipios vecinos.

In [ ]:
if not lista_seleccionados:
    raise ValueError(
        "La lista de vecinos está vacía. "
        "Agrega al menos un municipio en la sección 6."
    )

cvegeos_vec = [s["CVEGEO"] for s in lista_seleccionados]
vecinos = municipios[municipios["CVEGEO"].isin(cvegeos_vec)].copy()

print(f"Municipios vecinos confirmados: {len(vecinos)}")
vecinos[["CVE_ENT", "CVE_MUN", "NOM_ENT", "NOMGEO"]].reset_index(drop=True)

## 8. Visualización

In [ ]:
CRS_VIZ = "EPSG:3857"

mun_interes_plot = municipios.loc[[idx_interes]].to_crs(CRS_VIZ)
vecinos_plot     = vecinos.to_crs(CRS_VIZ)

fig, ax = plt.subplots(figsize=(12, 10))

vecinos_plot.plot(
    ax=ax, color="#5b8db8", alpha=0.6, edgecolor="white", linewidth=0.5
)
mun_interes_plot.plot(
    ax=ax, color="#c0392b", edgecolor="black", linewidth=1.0, zorder=5
)

for _, row in vecinos_plot.iterrows():
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    ax.annotate(
        row["NOMGEO"], xy=(cx, cy),
        fontsize=6.5, ha="center", va="center", color="#1a1a1a",
        bbox=dict(boxstyle="round,pad=0.15", fc="white", alpha=0.55, ec="none")
    )

try:
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom="auto")
except Exception:
    pass

leyenda = [
    mpatches.Patch(color="#c0392b",
                   label=f"Municipio de interés: {mun_interes['NOMGEO']}"),
    mpatches.Patch(color="#5b8db8",
                   label=f"Municipios vecinos ({len(vecinos)})"),
]
ax.legend(handles=leyenda, loc="lower left", fontsize=9, framealpha=0.9)
ax.set_title(
    f"Municipios vecinos de {mun_interes['NOMGEO']}, "
    f"{mun_interes['NOM_ENT']} (CVEGEO: {mun_interes['CVEGEO']})",
    fontsize=12, fontweight="bold"
)
ax.axis("off")
plt.tight_layout()

RUTA_SALIDA.mkdir(parents=True, exist_ok=True)
fig.savefig(
    RUTA_SALIDA / f"mapa_vecinos_{mun_interes['CVEGEO']}.png",
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Mapa guardado.")

## 9. Exportación de resultados

In [ ]:
import pandas as pd

prefijo   = mun_interes["CVEGEO"]

mun_export = municipios[mascara].copy()
mun_export["rol"] = "municipio_interes"

vec_export = vecinos.copy()
vec_export["rol"] = "vecino"

combinado = gpd.GeoDataFrame(
    pd.concat([mun_export, vec_export], ignore_index=True),
    geometry="geometry", crs=municipios.crs,
)

gpkg_path = RUTA_SALIDA / f"vecinos_{prefijo}.gpkg"
combinado.to_file(gpkg_path, layer="municipios", driver="GPKG")

csv_path = RUTA_SALIDA / f"vecinos_{prefijo}.csv"
(
    combinado[["CVEGEO", "CVE_ENT", "CVE_MUN", "NOM_ENT", "NOMGEO", "rol"]]
    .reset_index(drop=True)
    .to_csv(csv_path, index=False, encoding="utf-8-sig")
)

print(f"GeoPackage: {gpkg_path}")
print(f"CSV       : {csv_path}")
combinado[["CVEGEO", "CVE_ENT", "CVE_MUN", "NOM_ENT", "NOMGEO", "rol"]].reset_index(drop=True)